# parlvote to csv
extracts speeches from parlvote_full.csv and labels them by party ideology

In [ ]:
import pandas as pd
INPUT_FILE= '/kaggle/input/datasets/zainabrizvi5/thesis-data/ParlVote_full.csv'
OUTPUT_FILE= 'parlvote_labeled.csv'

In [ ]:
# party to ideology mapping
# conservative = right, labour = left, lib dem = center
# minor parties mapped to closest position
PARTY_TO_LABEL= {
    'conservative':'right',
    'labour':'left',
    'labourco-operative':'left',
    'liberal-democrat':'center',
    'scottish-national-party':'left',
    'plaid-cymru':'left',
    'social-democratic-and-labour-party':'left',
    'dup':'right',
    'uup':'right',
    'green':'left',
    'ukip':'right',
    'respect':'left',
    'alliance':'center',
}
# anything not in this list gets dropped
def map_party(party):
    if pd.isna(party):
        return None
    return PARTY_TO_LABEL.get(str(party).strip().lower(),None)

In [ ]:
df= pd.read_csv(INPUT_FILE, header=None, low_memory=False)
print(f'loaded {len(df)} rows, {len(df.columns)} columns')

In [ ]:
# extract motion speaker (col 3= party, col 5 = ext)
# and first response speaker (col 8 = party, col 10 = text)
rows= []
for _, row in df.iterrows():
    motion_party= map_party(row[3])
    motion_text= row[5]
    if motion_party and pd.notna(motion_text) and str(motion_text).strip():
        rows.append({'text':str(motion_text).strip(),'party': str(row[3]).strip().lower(),'label': motion_party,'word_count': len(str(motion_text).split()),'source':'motion'})
    response_party= map_party(row[8])
    response_text= row[10]
    if response_party and pd.notna(response_text) and str(response_text).strip():
        rows.append({'text': str(response_text).strip(),'party': str(row[8]).strip().lower(),'label': response_party,'word_count': len(str(response_text).split()),'source': 'response'})
print(f'extracted {len(rows)} speech rows')

In [ ]:
out= pd.DataFrame(rows, columns=['text','party','label','word_count','source'])
out.to_csv(OUTPUT_FILE, index=False)
print(f'saved to {OUTPUT_FILE}')
print(out['label'].value_counts())
print(out['word_count'].describe().round(1))